# Smart Agriculture – Crop Recommendation with Apache Spark MLlib

This notebook runs the project on Google Colab, so you do not need Ubuntu or a local Spark installation. It trains Decision Tree, Random Forest and Logistic Regression classifiers and compares their test accuracy.

**Dataset credit:** Atharva Ingle, [Crop Recommendation Dataset on Kaggle](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset).

In [ ]:
# Install Java and PySpark in this Colab runtime
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip -q install pyspark==3.5.1

import os, subprocess
java_home = subprocess.check_output(
    "dirname $(dirname $(readlink -f $(which java)))",
    shell=True, text=True
).strip()
os.environ["JAVA_HOME"] = java_home
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark environment ready")

In [ ]:
# Download this project's source code from your GitHub repository
!git clone https://github.com/deecares/Smart-Agriculture-Crop-Recommendation.git
%cd Smart-Agriculture-Crop-Recommendation
!ls

In [ ]:
# Download the CSV from a public mirror of the Kaggle dataset
# The original dataset attribution remains Kaggle – Atharva Ingle.
import os
os.makedirs("data", exist_ok=True)
!wget -q -O data/Crop_recommendation.csv https://raw.githubusercontent.com/the-amazing-atharva/Crop-Recommendation/main/Crop_recommendation.csv

import pandas as pd
df_preview = pd.read_csv("data/Crop_recommendation.csv")
print("Dataset shape:", df_preview.shape)
print("Columns:", list(df_preview.columns))
display(df_preview.head())

In [ ]:
# Run the Spark MLlib training and comparison pipeline
!spark-submit src/crop_recommendation.py --input data/Crop_recommendation.csv --output outputs/model_metrics

In [ ]:
# Display the saved model comparison and identify the measured winner
import glob
metric_files = glob.glob("outputs/model_metrics/part-*.csv")
if not metric_files:
    raise FileNotFoundError("Spark did not write a metrics CSV. Check the Spark output above.")
metrics = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
metrics = metrics.sort_values("test_accuracy", ascending=False).reset_index(drop=True)
display(metrics)
print("Best-performing model:", metrics.iloc[0]["model"])
print("Measured test accuracy:", round(float(metrics.iloc[0]["test_accuracy"]), 4))